# Phase 1 Exploration

Minimal examples for poll aggregation, sentiment velocity, feature generation, and edge scoring.

In [1]:
from datetime import UTC, datetime
from decimal import Decimal

from core import OrderBook, UnifiedMarket, Venue
from data.poll_aggregator import PollAggregator
from features import FeatureStore
from strategies import EdgeDetector

In [2]:
polls = PollAggregator().normalize_records(
    [
        {
            "event_slug": "demo-event",
            "pollster": "Example Polls",
            "grade": "A",
            "sample_size": 1000,
            "end_date": "2026-05-01T00:00:00+00:00",
            "answers": [{"candidate": "Yes", "support": 55}, {"candidate": "No", "support": 45}],
        }
    ]
)
aggregates = PollAggregator().aggregate(polls, as_of=datetime(2026, 5, 12, tzinfo=UTC))
aggregates

[PollAggregate(event_slug='demo-event', candidate='Yes', probability=0.55, as_of=datetime.datetime(2026, 5, 12, 0, 0, tzinfo=datetime.timezone.utc), market_id=None, poll_count=1, effective_sample_size=1032.6183455572914),
 PollAggregate(event_slug='demo-event', candidate='No', probability=0.45, as_of=datetime.datetime(2026, 5, 12, 0, 0, tzinfo=datetime.timezone.utc), market_id=None, poll_count=1, effective_sample_size=1032.6183455572914)]

In [3]:
market = UnifiedMarket(
    venue=Venue.POLYMARKET, market_id="demo", title="Demo market", outcomes=["Yes", "No"]
)
order_book = OrderBook(
    market_id="demo",
    bids=[(Decimal("0.48"), Decimal("100"))],
    asks=[(Decimal("0.50"), Decimal("80"))],
)
detector = EdgeDetector(FeatureStore())
signal = await detector.score_market(
    market, model_context={"poll_aggregates": aggregates, "order_book": order_book}
)
signal

2026-05-15 14:32:00 [info     ] features_built                 event_slug=None feature_count=19 market_id=demo
2026-05-15 14:32:00 [info     ] edge_signal_generated          advanced_adjustment=0.0 confidence=0.18860000000000027 edge=0.03750000000000009 market_id=demo market_prob=0.49 model_prob=0.5275000000000001


EdgeSignal(market_id='demo', market_prob=0.49, model_prob=0.5275000000000001, edge=0.03750000000000009, confidence=0.18860000000000027, reasoning=['model probability 0.528 vs market 0.490', 'poll top probability 0.550', 'sentiment 24h 0.000', 'cross-market conditional 0.500', 'spread 0.020'], features={'poll_top_probability': 0.55, 'poll_spread': 0.10000000000000003, 'poll_count': 2.0, 'poll_effective_sample_size': 2065.2366911145828, 'mention_count_24h': 0.0, 'mention_count_7d': 0.0, 'sentiment_24h': 0.0, 'sentiment_7d': 0.0, 'tone_shift': 0.0, 'price_divergence': 0.0, 'rolling_corr_7d': 0.0, 'rolling_corr_30d': 0.0, 'lead_lag_corr_1d': 0.0, 'implied_conditional_probability': 0.5, 'best_bid': 0.48, 'best_ask': 0.5, 'spread': 0.02, 'book_imbalance': 0.1111111111111111, 'top_book_liquidity': 180.0})